# 03 - 回测报告 / Backtest Report

端到端跑回测，绘制净值曲线、计算业绩指标，可选叠加事件 Agent。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from backtest import BacktestEngine, CostModel
from core.data import IngestService
from strategy import screen_pairs
from strategy.zscore_signal import SignalConfig

sys.path.insert(0, str(Path.cwd().parent))

svc = IngestService()
prices = svc.load_ohlcv()
with svc.engine.begin() as conn:
    stocks = pd.read_sql(text("SELECT * FROM stocks"), conn)
pairs = screen_pairs(prices, stocks, pvalue_threshold=0.10, max_pairs=10)
engine = BacktestEngine(cost_model=CostModel(), signal_cfg=SignalConfig())
result = engine.run(prices, pairs)
result.summary()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
result.equity.plot(ax=ax, color="steelblue")
ax.set_title("Equity curve")
plt.tight_layout()